In [ ]:
import os
import subprocess
import json
import requests
from pathlib import Path

def check_for_app(env):
    list_apps_url = f'{env["leonardo_url"]}/api/google/v1/apps/{env["google_project"]}'
    r = requests.get(
        list_apps_url,
        params={
          'includeDeleted': 'false'
        },
        headers = {
            'Authorization': f'Bearer {env["token"]}'
        }
    )
    r.raise_for_status()

    for potential_app in r.json():
        if potential_app['appType'] == 'CROMWELL' and (
                str(potential_app['auditInfo']['creator']) == env['owner_email']
                or str(potential_app['auditInfo']['creator']) == env['user_email']
        ) :
            potential_app_name = potential_app['appName']
            potential_app_status = potential_app['status']

            # We found a CROMWELL app in the correct google project and owned by the user. Now just check the workspace:
            _, workspace_namespace,  proxy_url = get_app_details(env, potential_app_name)
            if workspace_namespace == env['workspace_namespace']:
                return potential_app_name, potential_app_status, proxy_url['cromwell-service']

    return None, None, None

def get_app_details(env, app_name):
    get_app_url = f'{env["leonardo_url"]}/api/google/v1/apps/{env["google_project"]}/{app_name}'
    print('start')
    r = requests.get(
        get_app_url,
        params={
            'includeDeleted': 'true',
            'role': 'creator'
        },
        headers={
            'Authorization': f'Bearer {env["token"]}'
        }
    )
    if r.status_code == 404:
        return 'DELETED', None, None, None
    else:
        r.raise_for_status()
    result_json = r.json()
    custom_environment_variables = result_json['customEnvironmentVariables']
    return result_json['status'], custom_environment_variables['WORKSPACE_NAMESPACE'], result_json.get('proxyUrls')

# Checks that cromshell is installed. Otherwise raises an error.
def validate_cromshell():
    if validate_cromshell_alias():
        print("Found cromshell, please use cromshell")
    elif validate_cromshell_alpha():
        print("Found cromshell-alpha, please use cromshell-alpha")
    else:
        raise Exception("Cromshell is not installed.")

# Checks that cromshell is installed. Otherwise raises an error.
def validate_cromshell_alpha():
    print('Scanning for cromshell 2 alpha...')
    try:
        subprocess.run(['cromshell-alpha', 'version'], capture_output=True, check=True, encoding='utf-8')
    except FileNotFoundError:
        return False
    return True
# Checks that cromshell is installed. Otherwise raises an error.
def validate_cromshell_alias():
    print('Scanning for cromshell 2')
    try:
        subprocess.run(['cromshell', 'version'], capture_output=True, check=True, encoding='utf-8')
    except FileNotFoundError:
        return False
    return True

def configure_cromwell(env, proxy_url):
     print('Updating cromwell config')
     file = f'{str(Path.home())}/.cromshell/cromshell_config.json'
     configuration = {
        'cromwell_server': proxy_url.split("swagger/", 1)[0] if proxy_url else "invalid url",
        'requests_timeout': 5,
        'gcloud_token_email': env['user_email'],
        'referer_header_url': env['leonardo_url']
     }
     with open(file, 'w') as filetowrite:
        filetowrite.write(json.dumps(configuration, indent=2))

def find_app_status(env):
    print(f'Checking status for CROMWELL app')
    app_name, app_status, proxy_url = check_for_app(env)

    configure_cromwell(env, proxy_url)

    if app_name is None:
        print(f'CROMWELL app does not exist. Please create cromwell server from workbench')
    else:
        print(f'app_name={app_name}; app_status={app_status}')
        print(f'Existing CROMWELL app found (app_name={app_name}; app_status={app_status}).')
        exit(1)

def main():
    # Iteration 1: these ENV reads will throw errors if not set.
    env = {
        'workspace_namespace': os.environ['WORKSPACE_NAMESPACE'],
        'workspace_bucket': os.environ['WORKSPACE_BUCKET'],
        'user_email': os.environ.get('PET_SA_EMAIL', default = os.environ['OWNER_EMAIL']),
        'owner_email': os.environ['OWNER_EMAIL'],
        'google_project': os.environ['GOOGLE_PROJECT'],
        'leonardo_url': os.environ['LEONARDO_BASE_URL']
    }

    # Before going any further, check that cromshell2 is installed:
    validate_cromshell()

    # Fetch the token:
    token_fetch_command = subprocess.run(['gcloud', 'auth', 'print-access-token', env['user_email']], capture_output=True, check=True, encoding='utf-8')
    env['token'] = str.strip(token_fetch_command.stdout)

    find_app_status(env)


if __name__ == '__main__':
    main()
 


In [ ]:
import os
import pandas as pd
import subprocess

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
print(WORKSPACE_BUCKET)

In [ ]:

wdl_filename = "Himito_quickstart_PacBio.wdl"

WDL_content = """version 1.0

workflow Himito_quickstart{
    input {
        File whole_genome_bam
        File whole_genome_bai
        File reference_fa
        String prefix
        String sample_id
        String data_type
        Int kmer_size
        Float f
        String chromo
    }

    call QuickStart {
        input:
            bam = whole_genome_bam,
            bai = whole_genome_bai,
            reference_fa = reference_fa,
            prefix = prefix,
            kmer_size = kmer_size,
            f = f,
            sample_id = sample_id,
            chromo = chromo,
            data_type = data_type,

    }

    output {
        File graph = QuickStart.graph
        File methyl_bed = QuickStart.methyl_bed
        File asm = QuickStart.asm
        File read_var_mat = QuickStart.read_var_mat
        File read_methyl_mat = QuickStart.read_methyl_mat
        File numts_bam = QuickStart.numts_bam
        File vcf = QuickStart.vcf
    }
}

task QuickStart {
    input {
        File bam
        File bai
        File reference_fa
        String prefix
        Int kmer_size
        Float f
        String sample_id
        String data_type
        String chromo = "chrM"
    }

    command <<<
        set -euxo pipefail
        /Himito/target/release/Himito quick-start -i ~{bam} -c ~{chromo} -o ~{prefix} -k ~{kmer_size} -r ~{reference_fa} -s ~{sample_id} -d ~{data_type} --heteroplasmic-frequency-threshold ~{f}
    >>>  

    output {
        File graph = "~{prefix}.methyl.gfa"
        File methyl_bed = "~{prefix}.bed"
        File asm = "~{prefix}.fasta"
        File read_var_mat = "~{prefix}.matrix.csv"
        File read_methyl_mat = "~{prefix}.methylation_per_read.csv" 
        File numts_bam = "~{prefix}.numts.bam"
        File vcf = "~{prefix}.vcf"
    }

    runtime {
        docker: "us.gcr.io/broad-dsp-lrma/hangsuunc/himito:v1.1.0"
        memory: "4 GB"
        cpu: 1
        disks: "local-disk 200 SSD"
    }
}
"""
 
fp = open(wdl_filename, 'w')
fp.write(WDL_content)
fp.close()

In [ ]:
# Parameters
sample_id = "12345678"
bam = "gs://mybucket/myfile.bam"
bai = "gs://mybucket/myfile.bai"


In [ ]:
json_filename = "%s_Himito_quickstart_PacBio.json" % sample_id

json_content = """{
    "Himito_quickstart.whole_genome_bam": "%s",
    "Himito_quickstart.whole_genome_bai": "%s",
    "Himito_quickstart.reference_fa": "gs://fc-secure-2a2226a2-fac4-4539-ab85-590e33ea1774/NC_012920.1.fasta",
    "Himito_quickstart.prefix": "%s",
    "Himito_quickstart.sample_id": "%s",
    "Himito_quickstart.data_type": "pacbio",
    "Himito_quickstart.kmer_size": 21,
    "Himito_quickstart.f": 0.5,
    "Himito_quickstart.chromo": "chrM"
    
    }""" % (bam, bai, str(sample_id) + "_" + "Himito_pacbio", sample_id)

print(json_content)

#write to validate_vcf.json
fp = open(json_filename, 'w')
fp.write(json_content)
fp.close()

In [ ]:
!cromshell submit {wdl_filename} {json_filename}

In [ ]:
!cromshell status 4cda0783-44c7-47e8-8c93-b6f2f30bc632

In [ ]:
task_name = "Himito_quickstart"
submission_id = "4cda0783-44c7-47e8-8c93-b6f2f30bc632"
path = WORKSPACE_BUCKET + "/cromwell-execution/" + task_name + "/"  + submission_id + "/call-QuickStart/"
! gsutil ls $path